# Spark SQL

## Modern relational processing with DataFrames, Catalyst, Tungsten, and AQE

This notebook modernizes the supplied legacy slide deck into a Spark 3.5+ lesson. It focuses only on Spark SQL and current DataFrame workflows. Historical systems are mentioned only where they explain an obsolete API or design choice.

## Learning goals

You will learn to:

- use one `SparkSession` for DataFrame and SQL workloads;
- move between DataFrame transformations and SQL temporary views;
- understand parsed, analyzed, optimized, physical, and adaptive plans;
- understand what Python, Scala, Catalyst, generated Java, and JVM bytecode each do;
- inspect projection pruning, predicate pushdown, aggregation, joins, shuffles, caching, statistics, and AQE;
- replace legacy Spark SQL APIs with supported modern APIs;
- validate performance changes with plans and runtime metrics.

# 1. What Spark SQL is

Spark SQL is Spark's engine for structured and semi-structured data. It provides SQL text, the DataFrame API, catalog integration, built-in expressions, columnar file readers, and an optimizer/execution engine.

A DataFrame is a distributed dataset organized into named, typed columns. Transformations are lazy: they construct a logical plan. An action such as `show`, `count`, `collect`, or `write` starts execution. SQL and DataFrame APIs converge on the same Catalyst planning pipeline, so they can be freely composed.

## Legacy ideas removed or updated

| Legacy slide content | Modern Spark SQL treatment |
|---|---|
| `HiveContext()` | Use `SparkSession`; add `enableHiveSupport()` only when Hive catalog support is needed |
| `registerTempTable()` | Use `createOrReplaceTempView()` |
| RDD-to-DataFrame as the main entry point | Prefer `SparkSession.read`, `spark.table`, `spark.range`, or `createDataFrame` |
| DataFrame code builds an AST directly | It builds Catalyst expressions and logical-plan nodes; SQL text is parsed into a logical plan |
| SQL expressions are converted into Scala code | Catalyst is largely implemented in Scala, but whole-stage code generation emits Java source/JVM bytecode |
| Cost model only selects join algorithm | Modern CBO can estimate cardinality and reorder eligible joins; AQE can revise execution using runtime statistics |
| Historical query engines as primary context | Omitted; the lesson focuses on current Spark SQL |

# 2. Start Spark

Use the existing cluster session when your notebook platform supplies one. The builder below creates a local session for a standalone lab. AQE is enabled so Spark can refine parts of a physical plan after shuffle statistics become available.

In [ ]:
from pathlib import Path
import tempfile

from pyspark.sql import SparkSession, functions as F, types as T

builder = (
    SparkSession.builder
    .appName("Spark-SQL-Lab")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.shuffle.partitions", "16")
)

if SparkSession.getActiveSession() is None:
    builder = builder.master("local[*]")

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

# 3. Typed relational data

A schema supplies column names, data types, and nullability. Schema information allows Spark to resolve expressions, validate operations, select encodings, prune columns, generate code, and estimate memory more effectively than unstructured Python objects. Explicit schemas are preferable for production ingestion because inference requires work and can infer unwanted types.

In [ ]:
customer_schema = T.StructType(
    [
        T.StructField("customer_id", T.LongType(), nullable=False),
        T.StructField("customer_name", T.StringType(), nullable=False),
        T.StructField("country", T.StringType(), nullable=False),
        T.StructField("segment", T.StringType(), nullable=False),
    ]
)

customers = spark.createDataFrame(
    [
        (1, "Anika", "India", "VIP"),
        (2, "Bimal", "Nepal", "STANDARD"),
        (3, "Choden", "Bhutan", "STANDARD"),
        (4, "Dinesh", "India", "STANDARD"),
        (5, "Esha", "India", "VIP"),
    ],
    schema=customer_schema,
)

customers.printSchema()
customers.show(truncate=False)

# 4. DataFrame API and SQL are two front ends

The DataFrame API constructs unresolved Catalyst expressions through language bindings. SQL text goes through the SQL parser. After analysis, equivalent queries enter the same optimizer and physical planner. A temporary view stores a logical-plan reference for the session; creating it does not materialize the DataFrame.

In [ ]:
customers.createOrReplaceTempView("customers")

dataframe_query = (
    customers
    .where(F.col("country") == "India")
    .groupBy("segment")
    .agg(F.count("*").alias("customer_count"))
)

sql_query = spark.sql(
    """
    SELECT segment, COUNT(*) AS customer_count
    FROM customers
    WHERE country = 'India'
    GROUP BY segment
    """
)

dataframe_query.show()
sql_query.show()
print("Same optimized semantics:", dataframe_query.sameSemantics(sql_query))

# 5. The query lifecycle

![Modern Spark SQL planning lifecycle](assets/spark_sql_planning_lifecycle.svg)

```text
SQL text                         PySpark DataFrame calls
   |                                      |
SQL parser                       Py4J / Spark Connect protocol
   +------------------+-------------------+
                      v
             Unresolved logical plan
                      |
            Analyzer + catalog + types
                      v
              Analyzed logical plan
                      |
       Rule optimizer and optional CBO
                      v
              Optimized logical plan
                      |
        Physical planning and selection
                      v
              Initial physical plan
                      |
       execution / shuffle-stage metrics
                      v
             AQE may revise the plan
```

Catalyst uses immutable tree structures and rule batches. Rules transform matching subtrees until a batch reaches its stopping condition. Analysis resolves names and types; logical optimization preserves results while simplifying work; physical planning chooses executable operators.

## Read every plan view

- `simple`: physical plan only.
- `extended`: parsed, analyzed, optimized logical plans plus physical plan.
- `formatted`: compact operator tree plus per-node details.
- `cost`: optimized plan with estimated statistics.
- `codegen`: whole-stage generated code for supported physical operators.

Read a plan from the leaves upward: scans feed filters and projections, exchanges introduce distribution boundaries, and upper operators consume their results.

In [ ]:
print("EXTENDED PLAN")
dataframe_query.explain(mode="extended")

print("FORMATTED PHYSICAL PLAN")
dataframe_query.explain(mode="formatted")

print("COST ESTIMATES")
dataframe_query.explain(mode="cost")

# 6. Does PySpark code become Scala?

![PySpark, Catalyst, and JVM code-generation path](assets/pyspark_jvm_codegen.svg)

**No—not as a source-to-source translation.** Python UDFs, Python functions are not converted into JVM.

1. Python DataFrame calls create `Column` expressions and logical operators through Py4J in classic PySpark, or send a protocol plan in Spark Connect.
2. On the JVM, Spark represents the query as Catalyst trees. Catalyst and much of Spark SQL are implemented in Scala.
3. Catalyst analyzes and rewrites the trees. It does not convert the user's Python into equivalent Scala source.
4. Physical operators use Spark's JVM execution engine.
5. Whole-stage code generation can fuse compatible operators and generate Java source, which is compiled to JVM bytecode.
6. A regular Python UDF is different: its logic runs in Python workers and introduces a Python/JVM serialization boundary.

Therefore, built-in DataFrame expressions usually retain the same optimizer visibility and JVM execution advantages as SQL expressions.

In [ ]:
codegen_example = (
    spark.range(0, 1_000_000)
    .where(F.col("id") % 7 == 0)
    .select(
        F.col("id"),
        (F.col("id") * 10 + 1).alias("score"),
    )
    .groupBy(F.col("id") % 10)
    .agg(F.sum("score").alias("total_score"))
)

codegen_example.explain(mode="codegen")

## Whole-stage code generation in practical terms

Without operator fusion, execution repeatedly moves rows through generic operator interfaces. Whole-stage code generation combines compatible operators such as scan, filter, projection, and some aggregations into generated loops. This can reduce virtual calls, boxing, and intermediate objects.

In a physical plan, `*(n)` commonly marks operators participating in the same code-generation stage. Not every operator can be fused: exchanges, some joins, object operations, and Python execution boundaries separate stages. Generated code is an implementation detail; use `explain('codegen')` for learning and diagnosis rather than depending on its exact text.

# 7. Stage realistic Parquet data

File-backed data is required to observe column pruning, predicate pushdown, partition pruning, and scan metrics. The lab uses a dedicated temporary directory and generated rows, so it has no external dataset dependency.

In [ ]:
stage_root = tempfile.mkdtemp(prefix="spark_sql_lab_")
sales_path = f"{stage_root}/sales"
customer_path = f"{stage_root}/customer_dimension"

sales_generated = (
    spark.range(0, 500_000, numPartitions=16)
    .select(
        F.col("id").alias("sale_id"),
        (F.col("id") % 10_000).alias("customer_id"),
        F.date_add(
            F.lit("2025-01-01").cast("date"),
            (F.col("id") % 365).cast("int"),
        ).alias("sale_date"),
        F.round((F.col("id") % 500) * 1.17 + 5, 2).alias("amount"),
        F.element_at(
            F.array(F.lit("WEB"), F.lit("STORE"), F.lit("APP")),
            (F.col("id") % 3 + 1).cast("int"),
        ).alias("channel"),
    )
    .withColumn("sale_year", F.year("sale_date"))
    .withColumn("sale_month", F.month("sale_date"))
)

customer_dimension = (
    spark.range(0, 10_000)
    .select(
        F.col("id").alias("customer_id"),
        F.when(F.col("id") < 200, "VIP")
        .otherwise("STANDARD")
        .alias("segment"),
    )
)

(
    sales_generated.write
    .mode("overwrite")
    .partitionBy("sale_year", "sale_month")
    .parquet(sales_path)
)
customer_dimension.write.mode("overwrite").parquet(customer_path)

sales = spark.read.parquet(sales_path)
customer_dim = spark.read.parquet(customer_path)

# 8. Scan optimization

An efficient scan combines:

- **projection pruning:** read only required columns;
- **partition pruning:** skip directories using partition-column filters;
- **predicate pushdown:** pass supported data-column filters to Parquet;
- **vectorized reading:** decode columnar batches instead of individual objects.

In `formatted` output inspect `ReadSchema`, `PartitionFilters`, and `PushedFilters`. A residual Spark `Filter` may remain because pushed filters are not always sufficient to guarantee the final predicate semantics.

In [ ]:
scan_query = (
    sales
    .where(
        (F.col("sale_year") == 2025)
        & (F.col("sale_month") == 6)
        & (F.col("amount") >= 400)
    )
    .select("customer_id", "amount")
)

scan_query.explain(mode="formatted")

# 9. Aggregations and shuffles

Grouping by a key usually requires a shuffle so identical keys meet in the same partition. Spark commonly performs partial aggregation before the exchange and final aggregation afterward, reducing transferred records. `Exchange` is the key physical-plan marker.

`spark.sql.shuffle.partitions` is the initial partition count for SQL shuffles. Too few partitions create large tasks and spill; too many add scheduling and small-block overhead. AQE can coalesce small post-shuffle partitions, but measurement remains necessary.

In [ ]:
aggregation_query = (
    sales
    .where(F.col("sale_month").between(1, 3))
    .select("channel", "amount")
    .groupBy("channel")
    .agg(
        F.sum("amount").alias("revenue"),
        F.count("*").alias("sales_count"),
    )
)

aggregation_query.explain(mode="formatted")
aggregation_query.show(truncate=False)

# 10. Join planning

For equi-joins, common physical strategies include broadcast hash join, sort-merge join, and shuffled hash join. Broadcast hash join avoids shuffling the large side when the build side is safely small. Spark uses statistics, configuration, join type, hints, and—under AQE—runtime sizes.

A broadcast hint is a strong request, not a substitute for evidence. An unexpectedly large build side can exhaust driver or executor memory. Filter and project a dimension before broadcasting it.

In [ ]:
join_query = (
    sales
    .where(F.col("sale_month") == 6)
    .select("customer_id", "amount")
    .join(
        F.broadcast(customer_dim.select("customer_id", "segment")),
        "customer_id",
    )
    .groupBy("segment")
    .agg(F.sum("amount").alias("revenue"))
)

join_query.explain(mode="formatted")
join_query.show(truncate=False)

# 11. Statistics, CBO, and AQE

**CBO** means Cost-Based Optimizer. Before execution, it can use catalog table and column statistics to estimate cardinalities and compare eligible join orders. `ANALYZE TABLE` collects statistics for persistent catalog tables. Stale statistics can mislead planning.

**AQE** means Adaptive Query Execution. After execution starts, it uses materialized shuffle statistics to coalesce partitions, change some join strategies, and split certain skewed join partitions. CBO uses catalog estimates before execution; AQE uses observed evidence during execution.

`EXPLAIN COST` displays estimates. The Spark UI displays actual operator and stage metrics.

In [ ]:
aqe_query = (
    sales
    .join(customer_dim, "customer_id")
    .groupBy("segment", "sale_month")
    .agg(F.sum("amount").alias("revenue"))
)

print("BEFORE ACTION")
aqe_query.explain(mode="formatted")

aqe_query.collect()

print("AFTER ACTION")
aqe_query.explain(mode="formatted")

# 12. Cache only reused work

Persistence materializes a DataFrame after the first action and lets later actions reuse it. Cache an expensive, reused, reasonably narrow result—not every input. Caching a one-use DataFrame adds work. A cache is disposable runtime state, not durable storage or a correctness dependency.

In [ ]:
reused_sales = (
    sales
    .where(F.col("sale_month").between(1, 6))
    .select("customer_id", "channel", "amount")
    .cache()
)

reused_sales.count()
reused_sales.groupBy("channel").agg(F.sum("amount")).show()
reused_sales.groupBy("customer_id").agg(F.avg("amount")).show(5)
reused_sales.unpersist()

# 13. Built-ins versus Python UDFs

Prefer Spark SQL built-in expressions. Catalyst can inspect them, fold constants, push predicates, prune columns, and generate JVM code. A regular Python UDF is opaque to Catalyst and moves data through a Python worker boundary. Pandas UDFs use Arrow batches and may reduce overhead, but still restrict optimizer visibility. Use UDFs only when built-ins cannot express the requirement cleanly.

In [ ]:
built_in_expression = (
    sales
    .select(
        F.col("sale_id"),
        F.when(F.col("amount") >= 400, "HIGH")
        .when(F.col("amount") >= 150, "MEDIUM")
        .otherwise("LOW")
        .alias("value_band"),
    )
)

built_in_expression.explain(mode="codegen")

# 14. Performance workflow

1. Define an identical input and terminal action for the baseline and candidate.
2. Inspect `extended`, `formatted`, and `cost` plans.
3. Check scan schema, pushed filters, partition filters, exchanges, join strategy, and aggregate phases.
4. Run the query and inspect SQL/stage metrics: input bytes, output rows, shuffle bytes, spill, peak memory, task distribution, and duration.
5. Change one factor at a time. Account for warm filesystem caches and JVM/code-generation warm-up.
6. Validate correctness after every rewrite.

A shorter plan is not necessarily faster, and `explain()` alone does not execute or benchmark a query.

## Common technical gaps and corrections

- **Lazy does not mean cached:** transformations are deferred, but Spark may recompute lineage for each action.
- **DataFrame is not a local table:** rows are distributed; `collect()` moves all results to the driver.
- **Filter written early is not the full story:** Catalyst can reorder safe operations, but writing selective and narrow logic clearly still helps.
- **Partitions are overloaded terminology:** input partitions, shuffle partitions, and directory partition columns are different concepts.
- **Statistics are estimates:** an exact `count()` normally executes a job rather than trusting a potentially stale row count.
- **AQE does not fix everything:** it cannot repair incorrect joins, arbitrary Python bottlenecks, or historical small files.
- **SQL and DataFrame are not inherently faster than each other:** equivalent expressions normally converge to the same plan.
- **Generated code is not generated Scala:** modern whole-stage code generation targets Java/JVM bytecode.

# 15. End-to-end Spark SQL example

The example combines DataFrame staging with SQL querying. The view remains logical, allowing Catalyst to optimize across the view boundary.

In [ ]:
sales.createOrReplaceTempView("sales")
customer_dim.createOrReplaceTempView("customer_dimension")

final_query = spark.sql(
    """
    SELECT
        c.segment,
        s.channel,
        SUM(s.amount) AS revenue,
        COUNT(*) AS sales_count
    FROM sales AS s
    JOIN customer_dimension AS c
      ON s.customer_id = c.customer_id
    WHERE s.sale_year = 2025
      AND s.sale_month BETWEEN 4 AND 6
      AND s.amount >= 100
    GROUP BY c.segment, s.channel
    ORDER BY revenue DESC
    """
)

final_query.explain(mode="formatted")
final_query.show(truncate=False)

# Summary

Spark SQL unifies SQL text and typed DataFrame expressions through Catalyst logical plans. Analysis resolves names and types, optimizer rules simplify the plan, optional CBO uses catalog estimates, physical planning selects executable operators, whole-stage code generation can emit Java/JVM code, and AQE can revise remaining work using runtime statistics. Performance comes primarily from moving fewer bytes, choosing suitable joins and partitioning, avoiding opaque execution boundaries, and validating decisions with runtime evidence.

## References and further reading

- [Spark SQL reference](https://spark.apache.org/docs/3.5.6/sql-ref.html)
- [Spark SQL performance tuning](https://spark.apache.org/docs/3.5.6/sql-performance-tuning.html)
- [PySpark DataFrame API](https://spark.apache.org/docs/3.5.6/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.html)
- [Parquet data source](https://spark.apache.org/docs/3.5.6/sql-data-sources-parquet.html)

The supplied 2021 slide deck was used as historical source material. Obsolete product comparisons and APIs were intentionally removed or modernized.

## Optional cleanup

The local staged Parquet files are left available for plan inspection. Remove only the dedicated temporary directory after completing the lab.

In [ ]:
# Uncomment after confirming the generated directory name.
# import shutil
#
# assert Path(stage_root).name.startswith("spark_sql_lab_")
# shutil.rmtree(stage_root)
# spark.stop()